# LearnMate — M4 sufficiency-gate training
Runs on a free Colab **T4** (Runtime → Change runtime type → GPU). Checkpoints and caches are written to Google Drive so a disconnect never loses work.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!git clone -b claude/wonderful-bohr-1idsaa https://github.com/KirtirajChaudhari/LearnMate.git || (cd LearnMate && git pull)
%cd /content/LearnMate

In [ ]:
!pip -q install -e '.[gpu,eval,dev]'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# persist large artifacts on Drive (idempotent; also repairs broken links from older versions of this cell)
!python scripts/colab_setup.py

In [ ]:
!test -d data/raw/d2l-en || git clone https://github.com/d2l-ai/d2l-en.git data/raw/d2l-en
!git -C data/raw/d2l-en checkout -q 23d7a5aecceee57d1292c56e90cce307f183bb0a
!python -m learnmate.ingest --check
!python scripts/build_squad.py

Trains the DeBERTa-v3-base sufficiency gate on SQuAD 2.0 + BM25 hard negatives (spec C6) and the no-hard-negative ablation (A4). Resumable: rerun the same cell with `--resume` after a disconnect.

In [ ]:
!python -m learnmate.gate.train --out models/gate-v1 --resume

In [ ]:
!python -m learnmate.gate.train --out models/gate-noHN --no-hard-negatives --resume

In [ ]:
import json
for m in ['gate-v1','gate-noHN']:
    log=json.load(open(f'models/{m}/train_log.json'))
    print(m, {k:log[k] for k in ['best_dev_auroc','gpu_minutes','peak_vram_gb','examples_per_sec','steps']})